In [1]:
import pkg_resources
import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError
import mltable
from azure.ai.ml.entities import Data
from azure.ai.ml import automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.automl import ClassificationPrimaryMetrics
from azure.ai.ml import Input
from ucimlrepo import fetch_ucirepo
from mltable import DataType
import subprocess


C:\Users\leste\AppData\Local\Temp\ipykernel_5612\3143972866.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
subprocess.run("azd auth login")

CompletedProcess(args='azd auth login', returncode=0)

In [3]:

ml_client = MLClient(
    credential = DefaultAzureCredential(),
    subscription_id = "48f83b14-32a8-4911-b629-ea5aead7fb29",
    resource_group_name =  "lm1610-rg",
    workspace_name =  "lm_portfolio"
)

In [4]:
bank_marketing = fetch_ucirepo(id = 222)

In [5]:
X = bank_marketing.data.features
y = bank_marketing.data.targets


In [6]:
y.rename(columns = {
    'y' : 'Subscribed'
}, inplace = True)

C:\Users\leste\AppData\Local\Temp\ipykernel_5612\3286676455.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  y.rename(columns = {


In [7]:
bank_df = X.copy()
bank_df['y'] = y.copy()

In [8]:
bank_df

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,NaN,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,NaN,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,NaN,no


In [9]:
bank_updated_df = bank_df[~bank_df['job'].isnull()]

In [10]:
bank_updated_df.rename(columns = {
    "y" : 'Subscribed'
}, inplace = True)

C:\Users\leste\AppData\Local\Temp\ipykernel_5612\1331603712.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  bank_updated_df.rename(columns = {


In [11]:
bank_updated_df.drop(['education','poutcome','contact'], axis = 1,inplace = True)

C:\Users\leste\AppData\Local\Temp\ipykernel_5612\1124792161.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  bank_updated_df.drop(['education','poutcome','contact'], axis = 1,inplace = True)


In [12]:
bank_updated_df

,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,58,management,married,no,2143,yes,no,5,may,261,1,-1,0,no
1,44,technician,single,no,29,yes,no,5,may,151,1,-1,0,no
2,33,entrepreneur,married,no,2,yes,yes,5,may,76,1,-1,0,no
3,47,blue-collar,married,no,1506,yes,no,5,may,92,1,-1,0,no
5,35,management,married,no,231,yes,no,5,may,139,1,-1,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,no,825,no,no,17,nov,977,3,-1,0,yes
45207,71,retired,divorced,no,1729,no,no,17,nov,456,2,-1,0,yes
45208,72,retired,married,no,5715,no,no,17,nov,1127,5,184,3,yes
45209,57,blue-collar,married,no,668,no,no,17,nov,508,4,-1,0,no


In [13]:
bank_updated_df.to_csv(
    "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank-marketing.csv"
)

In [14]:
csv_path = [
    {
        "file" : "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank-marketing.csv"
    }
]

In [15]:
ml_tbl = mltable.from_delimited_files(paths = csv_path)

In [16]:
ml_tbl.show(10)

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,0,58,management,married,no,2143,yes,no,5,may,261,1,-1,False,no
1,1,44,technician,single,no,29,yes,no,5,may,151,1,-1,False,no
2,2,33,entrepreneur,married,no,2,yes,yes,5,may,76,1,-1,False,no
3,3,47,blue-collar,married,no,1506,yes,no,5,may,92,1,-1,False,no
4,5,35,management,married,no,231,yes,no,5,may,139,1,-1,False,no
5,6,28,management,single,no,447,yes,yes,5,may,217,1,-1,False,no
6,7,42,entrepreneur,divorced,yes,2,yes,no,5,may,380,1,-1,False,no
7,8,58,retired,married,no,121,yes,no,5,may,50,1,-1,False,no
8,9,43,technician,single,no,593,yes,no,5,may,55,1,-1,False,no
9,10,41,admin.,divorced,no,270,yes,no,5,may,222,1,-1,False,no


In [17]:
# converting column types

ml_tbl.convert_column_types(
    {
        'age' : DataType.to_int(),
        'balance' : DataType.to_float()
    })

paths:
- file: file://C:\Users\leste\OneDrive\Desktop\Microsoft VS Code\Github\MLOps-Portfolio\Data
    Files\bank-marketing.csv
transformations:
- read_delimited:
    delimiter: ','
    empty_as_string: false
    encoding: utf8
    header: all_files_same_headers
    include_path_column: false
    infer_column_types: true
    partition_size: 20971520
    path_column: Path
    support_multi_line: false
- convert_column_types:
  - column_type: int
    columns: age
  - column_type: float
    columns: balance
type: mltable

In [18]:
df_mltable = ml_tbl.to_pandas_dataframe()

> Filtering Data

In [19]:
df_mltable.columns

Index(['Column1', 'age', 'job', 'marital', 'default', 'balance', 'housing',
       'loan', 'day_of_week', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'Subscribed'],
      dtype='str')

In [20]:
# Select all columns and filter rows where age is greater than 40
df_mltable.filter(items = df_mltable.columns).filter(cond = df_mltable['age'] > 40)

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,0,58,management,married,no,2143,yes,no,5,may,261,1,-1,False,no
1,1,44,technician,single,no,29,yes,no,5,may,151,1,-1,False,no
3,3,47,blue-collar,married,no,1506,yes,no,5,may,92,1,-1,False,no
6,7,42,entrepreneur,divorced,yes,2,yes,no,5,may,380,1,-1,False,no
7,8,58,retired,married,no,121,yes,no,5,may,50,1,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44916,45204,73,retired,married,no,2850,no,no,17,nov,300,1,40,None,yes
44918,45206,51,technician,married,no,825,no,no,17,nov,977,3,-1,False,yes
44919,45207,71,retired,divorced,no,1729,no,no,17,nov,456,2,-1,False,yes
44920,45208,72,retired,married,no,5715,no,no,17,nov,1127,5,184,None,yes


In [21]:
# Select customers whose account balance is greater than 5000 and married
df_mltable.filter(cond = (df_mltable['balance'] > 5000) & (df_mltable['marital'] == 'married'))

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
33,34,51,management,married,no,10635,yes,no,5,may,336,1,-1,False,no
64,65,51,management,married,no,6530,yes,no,5,may,91,1,-1,False,no
69,70,57,blue-collar,married,no,5935,yes,yes,5,may,258,1,-1,False,no
204,205,50,services,married,no,5699,yes,no,5,may,381,2,-1,False,no
379,382,40,management,married,no,8486,no,no,6,may,260,3,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44832,45118,78,retired,married,no,14204,no,no,26,oct,150,1,183,True,no
44841,45127,67,blue-collar,married,no,16353,no,no,27,oct,223,2,-1,False,no
44844,45130,33,technician,married,no,5083,no,no,27,oct,161,1,96,None,no
44894,45181,46,blue-collar,married,no,6879,no,no,15,nov,74,2,118,None,no


In [22]:
# Finding high value customers 
df_mltable.filter(cond = (df_mltable['age'] > 30) & (df_mltable['balance'] >= 10000) & (df_mltable['housing'] == 'yes'))

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
33,34,51,management,married,no,10635,yes,no,5,may,336,1,-1,False,no
68,69,35,blue-collar,single,no,12223,yes,yes,5,may,177,1,-1,False,no
332,334,45,management,divorced,no,24598,yes,no,5,may,313,3,-1,False,no
443,446,39,technician,single,no,45248,yes,no,6,may,1623,1,-1,False,yes
916,920,44,admin.,married,no,58544,yes,no,7,may,144,2,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42703,42971,33,management,married,no,22867,yes,no,10,feb,165,1,92,None,no
43194,43465,40,technician,married,no,11766,yes,no,13,apr,218,2,242,None,yes
43516,43790,56,management,married,no,12356,yes,no,26,may,410,1,551,None,yes
44058,44335,36,services,married,no,11650,yes,no,29,jul,268,1,-1,False,yes


In [23]:
# Finding marketing oppurtunities
df_mltable.filter(cond = (df_mltable['loan'] == 'yes') & (df_mltable['Subscribed'] =='no'))

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
2,2,33,entrepreneur,married,no,2,yes,yes,5,may,76,1,-1,False,no
5,6,28,management,single,no,447,yes,yes,5,may,217,1,-1,False,no
19,20,28,blue-collar,married,no,723,yes,yes,5,may,262,1,-1,False,no
21,22,32,blue-collar,single,no,23,yes,yes,5,may,160,1,-1,False,no
23,24,40,retired,married,no,0,yes,yes,5,may,181,1,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44822,45108,36,technician,married,no,844,yes,yes,25,oct,129,1,-1,False,no
44836,45122,40,entrepreneur,single,no,262,yes,yes,26,oct,17,1,-1,False,no
44864,45151,47,management,single,no,311,yes,yes,9,nov,255,3,378,None,no
44866,45153,64,retired,married,no,2059,no,yes,9,nov,326,1,95,True,no


> Keeping & Dropping Columns

In [24]:
# Creating a customer profile using only relevant columns
customer_profile = ml_tbl.keep_columns(['age','job','marital','education','balance'])
customer_profile.to_pandas_dataframe()


,age,job,marital,balance
0,58,management,married,2143
1,44,technician,single,29
2,33,entrepreneur,married,2
3,47,blue-collar,married,1506
4,35,management,married,231
...,...,...,...,...
44918,51,technician,married,825
44919,71,retired,divorced,1729
44920,72,retired,married,5715
44921,57,blue-collar,married,668


> 1. Taking random samples 

> 2. Randomly splitting the data

> 3. Take `n` rows 

> 4. Skip `n` rows

In [25]:
sample_1 =  ml_tbl.take_random_sample(probability = 0.4 ,seed = 50)
sample_1.to_pandas_dataframe()

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,0,58,management,married,no,2143,yes,no,5,may,261,1,-1,False,no
1,2,33,entrepreneur,married,no,2,yes,yes,5,may,76,1,-1,False,no
2,10,41,admin.,divorced,no,270,yes,no,5,may,222,1,-1,False,no
3,12,53,technician,married,no,6,yes,no,5,may,517,1,-1,False,no
4,17,57,blue-collar,married,no,52,yes,no,5,may,38,1,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18103,45201,53,management,married,no,583,no,no,17,nov,226,1,184,None,yes
18104,45202,34,admin.,single,no,557,no,no,17,nov,224,1,-1,False,yes
18105,45203,23,student,single,no,113,no,no,17,nov,266,1,-1,False,yes
18106,45205,25,technician,single,no,505,no,yes,17,nov,386,2,-1,False,yes


In [26]:
sample_2 = ml_tbl.take_random_sample(probability = 0.3 , seed = 50)
sample_2.to_pandas_dataframe()

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,0,58,management,married,no,2143,yes,no,5,may,261,1,-1,False,no
1,10,41,admin.,divorced,no,270,yes,no,5,may,222,1,-1,False,no
2,12,53,technician,married,no,6,yes,no,5,may,517,1,-1,False,no
3,17,57,blue-collar,married,no,52,yes,no,5,may,38,1,-1,False,no
4,19,33,services,married,no,0,yes,no,5,may,54,1,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13645,45193,28,self-employed,single,no,159,no,no,16,nov,449,2,33,None,yes
13646,45201,53,management,married,no,583,no,no,17,nov,226,1,184,None,yes
13647,45202,34,admin.,single,no,557,no,no,17,nov,224,1,-1,False,yes
13648,45203,23,student,single,no,113,no,no,17,nov,266,1,-1,False,yes


In [27]:
ml_sample_test , ml_sample_train = ml_tbl.random_split(percent = 0.2 , seed = 123)
df_sample_train = ml_sample_train.to_pandas_dataframe()

In [28]:
df_sample_test = ml_sample_test.to_pandas_dataframe()

In [29]:
df_sample_train.isin(df_sample_test)

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,False,False,False,False,True,False,True,True,True,True,False,True,True,True,True
1,False,False,True,False,True,False,True,True,True,True,False,True,True,True,True
2,False,False,False,True,True,False,True,False,True,True,False,True,True,True,True
3,False,False,False,True,True,False,True,True,True,True,False,True,True,True,True
4,False,False,False,True,True,False,True,True,True,True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35783,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
35784,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
35785,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
35786,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [30]:
data_overlap = pd.merge(df_sample_test,df_sample_train,how = 'inner')
data_overlap # There is no data overlap

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed


In [31]:
ml_tbl.take(1000).to_pandas_dataframe() #Takesthe first 1000

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,0,58,management,married,no,2143,yes,no,5,may,261,1,-1,False,no
1,1,44,technician,single,no,29,yes,no,5,may,151,1,-1,False,no
2,2,33,entrepreneur,married,no,2,yes,yes,5,may,76,1,-1,False,no
3,3,47,blue-collar,married,no,1506,yes,no,5,may,92,1,-1,False,no
4,5,35,management,married,no,231,yes,no,5,may,139,1,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,999,33,technician,single,no,56,no,no,7,may,236,2,-1,False,no
996,1000,47,admin.,married,no,0,yes,no,7,may,164,1,-1,False,no
997,1001,50,technician,married,no,73,no,no,7,may,93,1,-1,False,no
998,1002,30,blue-collar,single,no,660,yes,no,7,may,912,2,-1,False,no


In [32]:
ml_tbl.skip(1000).to_pandas_dataframe() #skips the first 1000 and takes the rest 

,Column1,age,job,marital,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous,Subscribed
0,1004,44,blue-collar,divorced,no,558,yes,no,7,may,485,1,-1,False,no
1,1005,30,services,single,no,342,yes,no,7,may,206,2,-1,False,no
2,1006,40,blue-collar,married,no,-4,yes,no,7,may,311,1,-1,False,no
3,1007,41,blue-collar,married,no,225,yes,no,7,may,690,2,-1,False,no
4,1008,34,blue-collar,married,no,267,yes,no,7,may,362,1,-1,False,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43918,45206,51,technician,married,no,825,no,no,17,nov,977,3,-1,False,yes
43919,45207,71,retired,divorced,no,1729,no,no,17,nov,456,2,-1,False,yes
43920,45208,72,retired,married,no,5715,no,no,17,nov,1127,5,184,None,yes
43921,45209,57,blue-collar,married,no,668,no,no,17,nov,508,4,-1,False,no


In [33]:
ml_df = ml_tbl

In [34]:
try:
    ml_tbl.save('C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank_data_mlt/MLTable')
except Exception as e:
    print('The file is already saved in the folder')

The file is already saved in the folder


In [35]:
file_path = csv_path[0]['file']
ml_tbl_path = r'C:\Users\leste\OneDrive\Desktop\Microsoft VS Code\Github\MLOps-Portfolio\Data Files\bank_data_mlt'


In [49]:
# ML table defination
bank_data_ml_table_def = Data(
    name = 'Bank-MLtable',
    version = '1',
    description = 'This dataset has been extracted from the UCIML repo and converted to MLtable',
    path = ml_tbl_path,
    type = AssetTypes.MLTABLE
)

In [50]:
ml_client.data.create_or_update(data = bank_data_ml_table_def)


Data({'path': 'azureml://subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourcegroups/lm1610-rg/workspaces/lm_portfolio/datastores/workspaceblobstore/paths/LocalUpload/0808ca7a67d6dbc6663a7abb639fb7e45e70157be1712a58d498e3f8cf640fe3/bank_data_mlt/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['../bank-marketing.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'Bank-MLtable', 'description': 'This dataset has been extracted from the UCIML repo and converted to MLtable', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/data/Bank-MLtable/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\leste\\OneDrive\\Desktop\\Microsoft VS Code\\Github\\MLOps-Portfolio', 'creation_context': <azure.ai.ml.entities._system_data.Syste

In [63]:
bank_mltable_asset = ml_client.data.get('Bank-MLtable', version = '1')
bank_mltable_asset.path

'azureml://subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourcegroups/lm1610-rg/workspaces/lm_portfolio/datastores/workspaceblobstore/paths/LocalUpload/0808ca7a67d6dbc6663a7abb639fb7e45e70157be1712a58d498e3f8cf640fe3/bank_data_mlt/'

In [70]:
import mltable

tbl = mltable.load(bank_mltable_asset.path)

print(tbl)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


paths:
- file: azureml://subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourcegroups/lm1610-rg/workspaces/lm_portfolio/datastores/workspaceblobstore/paths/LocalUpload/0808ca7a67d6dbc6663a7abb639fb7e45e70157be1712a58d498e3f8cf640fe3/bank_data_mlt/..\bank-marketing.csv
transformations:
- read_delimited:
    delimiter: ','
    empty_as_string: false
    encoding: utf8
    header: all_files_same_headers
    include_path_column: false
    infer_column_types: true
    partition_size: 20971520
    path_column: Path
    support_multi_line: false
type: mltable



In [72]:
tbl.show(10)

Message: rslex failed
Payload: {"pid": 5612, "MLTableVersion": "1.6.3", "rslex_version": "2.24.6", "version": "5.3.2"}


UserErrorException: 
Error Code: ScriptExecution.StreamAccess.NotFound
Native Error: Dataflow visit error: ExecutionError(StreamError(NotFound))
	VisitError(ExecutionError(StreamError(NotFound)))
=> Failed with execution error: error in streaming from input data sources
	ExecutionError(StreamError(NotFound))
Error Message: The requested stream was not found. Please make sure the request uri is correct.| session_id=l_f0023b2e-297c-497c-ad97-b90c00975944

In [53]:
bank_ml_table = Input(
    type = AssetTypes.MLTABLE,
    path = bank_mltable_asset.id,
    mode = 'download'
)

In [54]:
bank_ml_table

{'type': 'mltable', 'path': '/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/data/Bank-MLtable/versions/1', 'mode': 'download'}

In [60]:
classification_job = automl.classification(
    name = 'UCML-classification-v4',
    training_data = bank_ml_table,
    experiment_name = 'test_classification_experiment',
    target_column_name = 'Subscribed',
    primary_metric = 'accuracy', # The column to observe for performance 
    n_cross_validations = 5 ,
    validation_data_size = 0.3,
    cv_split_column_names = ['Fold_1', 'Fold_2', 'Fold_3', 'Fold_4', 'Fold_5']

)

In [61]:
classification_job_limitations = classification_job.set_limits(
    enable_early_termination = True,
    max_trials = 6,
    timeout_minutes = 45,
    trial_timeout_minutes = 20
)

In [62]:
return_classification_job = ml_client.jobs.create_or_update(classification_job)

In [ ]:
list(ClassificationPrimaryMetrics)